# M1 — Validación de Detección de Placas (YOLOv9-t ONNX) + OCR Híbrido
Clona el repositorio de GitHub (ya incluye los pesos ONNX de M1) y evalúa **fast-plate-ocr**, **EasyOCR** y el pipeline **Híbrido boliviano**.

In [1]:
!nvidia-smi -L || echo "Ejecutando en CPU"
!pip install -q onnxruntime-gpu==1.32.0 fast-plate-ocr==1.1.0 easyocr==1.7.2 roboflow==1.7.4 opencv-python-headless==5.1.0.95 pandas matplotlib
!rm -rf /content/repo_peaje && git clone -q https://github.com/Cristopher-boop/iam811-peaje-inteligente.git /content/repo_peaje

GPU 0: NVIDIA A40 (UUID: GPU-e5aa9d4d-cd18-12ae-f2a6-9724225d43f6)


In [2]:
import os, re, time, json
from pathlib import Path
from getpass import getpass
import cv2
import numpy as np
import pandas as pd
import onnxruntime as ort
from fast_plate_ocr import LicensePlateRecognizer
import easyocr
from roboflow import Roboflow

MODEL_DIR = Path('/content/repo_peaje/inferencia/models')
DET_PATH = MODEL_DIR / 'yolo-v9-t-384-license-plates-end2end.onnx'
OCR_PATH = MODEL_DIR / 'cct_xs_v2_global.onnx'
OCR_CFG = MODEL_DIR / 'cct_xs_v2_global_plate_config.yaml'
assert DET_PATH.exists() and OCR_PATH.exists(), 'No se encontraron los modelos ONNX en el repo'

LADO = 384
FORMATO_BOLIVIA = re.compile(r'^\d{3,4}[A-Z]{3}$')
PERMITIDOS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
sesion_det = ort.InferenceSession(str(DET_PATH), providers=providers)
lector_fast = LicensePlateRecognizer(onnx_model_path=str(OCR_PATH), plate_config_path=str(OCR_CFG))
lector_easy = easyocr.Reader(['en'], gpu='CUDAExecutionProvider' in providers, verbose=False)
print('M1 inicializado con providers:', providers)

*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:729 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(PySessionOptions&, const onnxruntime::ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************


2026-11-25 17:29:30.379726544 [E:onnxruntime:Default, provider_bridge_ort.cc:2422 operator()] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:2059 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /workspace/venv_peaje/lib/python3.12/site-packages/onnxruntime/capi/libonnxruntime_providers_tensorrt.so with error: libnvinfer.so.10: cannot open shared object file: No such file or directory



M1 inicializado con providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']


In [3]:
def limpiar_placa(texto: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', (texto or '').upper())

def detectar_placas(imagen_bgr, umbral=0.35):
    alto, ancho = imagen_bgr.shape[:2]
    escala = min(LADO / alto, LADO / ancho)
    nw, nh = round(ancho * escala), round(alto * escala)
    mx, my = (LADO - nw) / 2, (LADO - nh) / 2
    lienzo = cv2.resize(imagen_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    lienzo = cv2.copyMakeBorder(lienzo, round(my - 0.1), round(my + 0.1), round(mx - 0.1), round(mx + 0.1), cv2.BORDER_CONSTANT, value=(114, 114, 114))
    entrada = (lienzo[:, :, ::-1].transpose(2, 0, 1) / 255.0).astype(np.float32)[None]
    salida = sesion_det.run(None, {sesion_det.get_inputs()[0].name: entrada})[0]
    cajas = []
    for _, x1, y1, x2, y2, _, conf in salida:
        if conf < umbral:
            continue
        x1, x2 = [(v - mx) / escala for v in (x1, x2)]
        y1, y2 = [(v - my) / escala for v in (y1, y2)]
        cajas.append((int(max(0, x1)), int(max(0, y1)), int(min(ancho, x2)), int(min(alto, y2)), float(conf)))
    cajas.sort(key=lambda c: -c[4])
    return cajas

def leer_ocr_fast(recorte_bgr):
    rgb = cv2.cvtColor(recorte_bgr, cv2.COLOR_BGR2RGB)
    pred = lector_fast.run_one(rgb, return_confidence=True)
    return limpiar_placa(pred.plate), float(pred.char_probs.mean()) if pred.char_probs is not None else 0.0

def leer_ocr_easy(recorte_bgr):
    lecturas = lector_easy.readtext(recorte_bgr, allowlist=PERMITIDOS)
    if lecturas:
        altura = lambda l: max(p[1] for p in l[0]) - min(p[1] for p in l[0])
        hmax = max(altura(l) for l in lecturas)
        lecturas = [l for l in lecturas if altura(l) >= hmax * 0.6]
    lecturas.sort(key=lambda l: l[0][0][0])
    txt = limpiar_placa(''.join(t for _, t, _ in lecturas))
    conf = float(sum(c for _, _, c in lecturas) / len(lecturas)) if lecturas else 0.0
    return txt, conf

def leer_ocr_hibrido(recorte_bgr):
    tf, cf = leer_ocr_fast(recorte_bgr)
    te, ce = leer_ocr_easy(recorte_bgr)
    if FORMATO_BOLIVIA.match(te) and not FORMATO_BOLIVIA.match(tf):
        return te, ce, 'easyocr'
    if FORMATO_BOLIVIA.match(tf) and not FORMATO_BOLIVIA.match(te):
        return tf, cf, 'fast-plate-ocr'
    if ce > cf and len(te) >= 5:
        return te, ce, 'easyocr'
    return (tf, cf, 'fast-plate-ocr') if len(tf) >= len(te) else (te, ce, 'easyocr')

In [4]:
# Descarga dataset liviano (~140 MB) con placas reales anotadas en autopista
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY') or getpass('Pega tu Roboflow Private API Key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
ds_placas = rf.workspace('helmet-o8tke').project('vehicle-segmentation-pyt1u').version(1).download('yolov11', location='/content/test_placas')
DIR_VALID = Path(ds_placas.location) / 'valid'
import yaml as _yaml
_names = _yaml.safe_load((Path(ds_placas.location) / 'data.yaml').read_text()).get('names', [])
_pid = _names.index('license-plate') if isinstance(_names, list) else next(k for k, v in _names.items() if v == 'license-plate')
imgs = []
for sp in ('valid', 'test', 'train'):
    for p in sorted((Path(ds_placas.location) / sp / 'images').glob('*')):
        lp = p.parent.parent / 'labels' / f'{p.stem}.txt'
        if lp.exists() and any(int(float(ln.split()[0])) == _pid for ln in lp.read_text().splitlines() if ln.strip()):
            imgs.append(p)
        if len(imgs) >= 50:
            break
    if len(imgs) >= 50:
        break

filas = []
for p in imgs:
    img = cv2.imread(str(p))
    if img is None:
        continue
    t0 = time.perf_counter()
    cajas = detectar_placas(img)
    dt_det = (time.perf_counter() - t0) * 1000
    if not cajas:
        filas.append({'archivo': p.name, 'detectada': False, 'fast': '', 'easy': '', 'hibrido': '', 'conf_ocr': 0.0, 'ms': round(dt_det, 1)})
        continue
    x1, y1, x2, y2, cdet = cajas[0]
    crop = img[y1:y2, x1:x2]
    t1 = time.perf_counter()
    tf, cf = leer_ocr_fast(crop)
    te, ce = leer_ocr_easy(crop)
    th, ch, ganador = leer_ocr_hibrido(crop)
    dt_tot = dt_det + (time.perf_counter() - t1) * 1000
    filas.append({'archivo': p.name, 'detectada': True, 'conf_det': round(cdet, 3), 'fast': tf, 'easy': te, 'hibrido': th, 'ganador': ganador, 'conf_ocr': round(ch, 3), 'ms': round(dt_tot, 1)})

df_res = pd.DataFrame(filas)
display(df_res.head(20))
resumen = {
    'total_imagenes': int(len(df_res)),
    'tasa_deteccion_placa': round(float(df_res['detectada'].mean()), 4) if len(df_res) else 0,
    'tasa_lectura_no_vacia': round(float((df_res['hibrido'].str.len() >= 4).mean()), 4) if len(df_res) else 0,
    'confianza_ocr_promedio': round(float(df_res.loc[df_res['detectada'], 'conf_ocr'].mean()), 4) if df_res['detectada'].any() else 0,
    'latencia_promedio_ms': round(float(df_res['ms'].mean()), 2) if len(df_res) else 0,
}
print('=== METRICAS M1 ===')
print(json.dumps(resumen, indent=2, ensure_ascii=False))
Path('/content/resumen_metricas_m1.json').write_text(json.dumps(resumen, indent=2, ensure_ascii=False))

loading Roboflow workspace...


loading Roboflow project...


Extracting Dataset Version Zip to /content/test_placas in yolov11::   0%|          | 0/6091 [00:00<?, ?it/s]

Extracting Dataset Version Zip to /content/test_placas in yolov11::  10%|█         | 614/6091 [00:00<00:00, 6134.81it/s]

Extracting Dataset Version Zip to /content/test_placas in yolov11::  21%|██        | 1253/6091 [00:00<00:00, 6283.67it/s]

Extracting Dataset Version Zip to /content/test_placas in yolov11::  31%|███       | 1891/6091 [00:00<00:00, 6324.44it/s]

... (6 outputs left out)

Extracting Dataset Version Zip to /content/test_placas in yolov11:: 100%|██████████| 6091/6091 [00:00<00:00, 6200.15it/s]

,archivo,detectada,conf_det,fast,easy,hibrido,ganador,conf_ocr,ms
0,bus_vid2_frame_606_jpg.rf.4b01e4857ca624455e3b...,True,0.866,AE335,,AE335,fast-plate-ocr,0.595,880.0
1,swift_vid5_frame_397_jpg.rf.370b19e447c1b80047...,True,0.880,1142VH4,,1142VH4,fast-plate-ocr,0.496,55.6
2,van_vid11_frame_2241_jpg.rf.be0c835b03d771c165...,True,0.856,AJ46EEG,,AJ46EEG,fast-plate-ocr,0.463,53.7
3,vid10_frame_88_jpg.rf.70ea182db355dbb649f236e6...,True,0.676,1A1111,,1A1111,fast-plate-ocr,0.452,55.5
4,vid10_frame_9_jpg.rf.5f2750c6d39da7b5086a8f891...,True,0.604,LL11T,,LL11T,fast-plate-ocr,0.726,23.4
5,vid11_frame_1120_jpg.rf.0cc8ae1f372dcc863a1eb2...,True,0.598,11LL11,,11LL11,fast-plate-ocr,0.507,21.6
6,vid11_frame_1200_jpg.rf.41c0413ca196658eaa4b2f...,True,0.742,VTV559,,VTV559,fast-plate-ocr,0.580,21.4
7,vid11_frame_1253_jpg.rf.1ddd6d5d8b75857cd4cc38...,False,NaN,,,,NaN,0.000,8.3
8,vid11_frame_1362_jpg.rf.710dee7cd7e8a267a2f56d...,False,NaN,,,,NaN,0.000,8.2
9,vid11_frame_1483_jpg.rf.9bca11d7bab8b05c119fbf...,False,NaN,,,,NaN,0.000,8.0


=== METRICAS M1 ===
{
  "total_imagenes": 50,
  "tasa_deteccion_placa": 0.48,
  "tasa_lectura_no_vacia": 0.38,
  "confianza_ocr_promedio": 0.6054,
  "latencia_promedio_ms": 33.76
}


160